# Backtest

Phase 4–6: load cached signal events and wallet cohorts, run a full sweep of all strategy combinations (cohorts × trigger rules × sizing modes), display a summary table, then plot top-N strategies + placebo benchmark.

In [12]:
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.dataset as ds


In [13]:
# ── configuration ─────────────────────────────────────────────────────────────
# END_DATE_TRAIN and TRAIN_A_END_DATE are derived from the data below

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]
FILL_MAX_REL_PRICE_DIFF_BY_BUCKET = {
    "0.00-0.10": 2.50,
    "0.10-0.25": 1.20,
    "0.25-0.40": 0.70,
    "0.40-0.60": 0.35,
    "0.60-0.75": 0.22,
    "0.75-0.90": 0.12,
    "0.90-1.00": 0.06,
}

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

CALIBRATION_SIGNALS_PATH  = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH          = WORKSPACE_DIR / "signal_events_test.parquet"
METRICS_FULL_PATH          = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
SELECTED_WALLETS_PATH      = WORKSPACE_DIR / "selected_wallets_v2.parquet"
EXEC_TAPE_TEST_PATH        = WORKSPACE_DIR / "execution_tape_test_v2.parquet"
EXEC_TAPE_TRAIN_B_PATH     = WORKSPACE_DIR / "execution_tape_train_b_v2.parquet"

COHORT_TOP_N = 100


In [14]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
TRAIN_A_END_DATE = TRAIN_START_DATE + (END_DATE_TRAIN - TRAIN_START_DATE) // 2
del _sample
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-03-07  TRAIN_A_END_DATE=2026-02-16


In [15]:
# ── load cached metrics and selected wallets ──────────────────────────────────
full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
selected_wallets   = pd.read_parquet(SELECTED_WALLETS_PATH)

train_b_open_buys  = pd.read_parquet(CALIBRATION_SIGNALS_PATH)
test_open_buys     = pd.read_parquet(TEST_SIGNALS_PATH)
print(f"train_b signals: {len(train_b_open_buys):,}  test signals: {len(test_open_buys):,}")


train_b signals: 2,188  test signals: 2,936


In [16]:
# ── rebuild calibration tables and pick threshold ─────────────────────────────
from polymarket_analysis.signal.scorer import (
    build_calibration_tables,
    apply_signal_score,
    select_signal_threshold,
)

price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
print(f"Global edge: {global_edge:.4f}")
print(f"Signal threshold: {SIGNAL_THRESHOLD:.2f}")

Global edge: 0.1447
Signal threshold: 0.80


In [17]:
# ── build wallet cohorts and score test signals per cohort ────────────────────
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts
from polymarket_analysis.signal.builder import build_wallet_profiles, build_signal_events

wallet_cohorts = build_wallet_cohorts(
    full_train_metrics, train_b_open_buys, selected_wallets, top_n=COHORT_TOP_N
)

cohort_test_signals      = {}
cohort_train_ref_signals = {}
for cohort_name, cohort_wallets in wallet_cohorts.items():
    profiles = build_wallet_profiles(
        dataset, cohort_wallets, period="full_train",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
    )
    _, c_test  = build_signal_events(
        dataset, profiles, period="test",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    _, c_train = build_signal_events(
        dataset, profiles, period="train_b",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    cohort_test_signals[cohort_name]      = apply_signal_score(c_test,  price_table, consensus_table)
    cohort_train_ref_signals[cohort_name] = apply_signal_score(c_train, price_table, consensus_table)

print({name: len(df) for name, df in cohort_test_signals.items()})


{'quality_core': 3, 'early_entry': 0, 'smooth_pnl': 137}


In [18]:
# ── build / load execution tapes ──────────────────────────────────────────────
from polymarket_analysis.backtest.execution_tape import (
    build_execution_tape,
    build_tape_groups,
    normalize_execution_tape,
)

def _collect_cids(signals_dict):
    parts = [df["condition_id"] for df in signals_dict.values() if not df.empty]
    if not parts:
        return []
    return list(pd.concat(parts, ignore_index=True).drop_duplicates())

all_test_cids  = _collect_cids(cohort_test_signals)
all_train_cids = _collect_cids(cohort_train_ref_signals)

if EXEC_TAPE_TEST_PATH.exists():
    execution_tape_test = pd.read_parquet(EXEC_TAPE_TEST_PATH)
else:
    execution_tape_test = build_execution_tape(dataset, all_test_cids, tx_hash_column=TRIGGER_TX_HASH_COLUMN)
    execution_tape_test.to_parquet(EXEC_TAPE_TEST_PATH, index=False)

if EXEC_TAPE_TRAIN_B_PATH.exists():
    execution_tape_train_b = pd.read_parquet(EXEC_TAPE_TRAIN_B_PATH)
else:
    execution_tape_train_b = build_execution_tape(
        dataset, all_train_cids,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
        start_date=TRAIN_A_END_DATE + datetime.timedelta(days=1),
        end_date=END_DATE_TRAIN,
    )
    execution_tape_train_b.to_parquet(EXEC_TAPE_TRAIN_B_PATH, index=False)

execution_tape_test["tape_dt"]    = pd.to_datetime(execution_tape_test["tape_dt"],    utc=True)
execution_tape_train_b["tape_dt"] = pd.to_datetime(execution_tape_train_b["tape_dt"], utc=True)
tape_groups           = build_tape_groups(normalize_execution_tape(execution_tape_test))
tape_groups_train_ref = build_tape_groups(normalize_execution_tape(execution_tape_train_b))
print(f"tape markets – test: {len(tape_groups):,}  train_b: {len(tape_groups_train_ref):,}")


tape markets – test: 30,258  train_b: 89,673


In [19]:
# ── full strategy sweep ───────────────────────────────────────────────────────
from polymarket_analysis.backtest.strategy import (
    backtest_strategy,
    summarize_backtest,
    build_trigger_ledger,
)

def _build_specs(cohort_signals, threshold):
    specs = []
    for cohort_name, scored_df in cohort_signals.items():
        if scored_df.empty:
            continue
        specs += [
            {
                "name": f"{cohort_name}__score_threshold_{threshold:.2f}",
                "cohort": cohort_name,
                "signals": scored_df,
                "mask": scored_df["signal_score"] >= threshold,
                "trigger_rule": f"signal_score >= {threshold:.2f}",
                "dynamic_sizing": True,
            },
            {
                "name": f"{cohort_name}__all_open_buys",
                "cohort": cohort_name,
                "signals": scored_df,
                "mask": scored_df["wallet"].notna(),
                "trigger_rule": "all selected-wallet open_buy events",
                "dynamic_sizing": False,
            },
        ]
    return specs

BACKTEST_KWARGS = dict(
    base_stake_usdc=100.0,
    latency_seconds=0,
    fill_horizon_seconds=600,
    slippage_bps=50.0,
    fee_bps=10.0,
    max_signals_per_day=20,
    dedupe_by_market=True,
    starting_capital=10_000.0,
    max_rel_price_diff_by_bucket=FILL_MAX_REL_PRICE_DIFF_BY_BUCKET,
)

def _run_specs(specs, tapes):
    runs, rows = {}, []
    for spec in specs:
        trades_bt, daily_bt, unfilled_bt, theoretical_bt = backtest_strategy(
            spec["signals"], spec["mask"], tapes,
            spec["name"], spec["trigger_rule"],
            dynamic_sizing=spec["dynamic_sizing"],
            cohort_name=spec["cohort"],
            **BACKTEST_KWARGS,
        )
        runs[spec["name"]] = {
            "trades": trades_bt, "daily": daily_bt,
            "unfilled_triggers": unfilled_bt, "theoretical_daily": theoretical_bt,
            "spec": spec,
        }
        row = summarize_backtest(trades_bt, daily_bt).to_dict()
        row.update({
            "strategy": spec["name"],
            "cohort": spec["cohort"],
            "trigger_rule": spec["trigger_rule"],
            "dynamic_sizing": spec["dynamic_sizing"],
            "triggered_signals": len(trades_bt) + len(unfilled_bt),
            "unfilled_triggers": len(unfilled_bt),
            "fill_rate": len(trades_bt) / (len(trades_bt) + len(unfilled_bt)) if (len(trades_bt) + len(unfilled_bt)) else np.nan,
        })
        rows.append(row)
    if not rows:
        return runs, pd.DataFrame()
    summary = pd.DataFrame(rows).set_index("strategy").sort_values("net_roi_on_stake", ascending=False)
    return runs, summary

strategy_runs,           strategy_summary           = _run_specs(_build_specs(cohort_test_signals,      SIGNAL_THRESHOLD), tape_groups)
strategy_runs_train_ref, strategy_summary_train_ref = _run_specs(_build_specs(cohort_train_ref_signals, SIGNAL_THRESHOLD), tape_groups_train_ref)

trigger_ledgers   = {n: build_trigger_ledger(r["trades"], r["unfilled_triggers"]) for n, r in strategy_runs.items()}
all_trigger_ledger = pd.concat([l for l in trigger_ledgers.values() if not l.empty], ignore_index=True) if trigger_ledgers else pd.DataFrame()

strategy_summary


,filled_trades,net_pnl_usdc,net_roi_on_stake,win_rate,cohort,trigger_rule,dynamic_sizing,triggered_signals,unfilled_triggers,fill_rate,avg_signal_score,avg_kelly_fraction,avg_fill_delay_minutes,opposite_fill_share,max_drawdown_usdc
strategy,,,,,,,,,,,,,,,
smooth_pnl__score_threshold_0.80,1.0,524.787085,1.071017,1.0,smooth_pnl,signal_score >= 0.80,True,25,24,0.040000,0.812524,0.048999,2.566667,0.000000,0.0
smooth_pnl__all_open_buys,6.0,-71.389244,-0.118982,0.5,smooth_pnl,all selected-wallet open_buy events,False,55,49,0.109091,0.587807,NaN,2.538889,0.207933,0.0
quality_core__score_threshold_0.80,0.0,0.000000,NaN,NaN,quality_core,signal_score >= 0.80,True,1,1,0.000000,NaN,NaN,NaN,NaN,NaN
quality_core__all_open_buys,0.0,0.000000,NaN,NaN,quality_core,all selected-wallet open_buy events,False,3,3,0.000000,NaN,NaN,NaN,NaN,NaN


## Summary table

In [20]:
pd.set_option("display.max_columns", None)
strategy_summary


,filled_trades,net_pnl_usdc,net_roi_on_stake,win_rate,cohort,trigger_rule,dynamic_sizing,triggered_signals,unfilled_triggers,fill_rate,avg_signal_score,avg_kelly_fraction,avg_fill_delay_minutes,opposite_fill_share,max_drawdown_usdc
strategy,,,,,,,,,,,,,,,
smooth_pnl__score_threshold_0.80,1.0,524.787085,1.071017,1.0,smooth_pnl,signal_score >= 0.80,True,25,24,0.040000,0.812524,0.048999,2.566667,0.000000,0.0
smooth_pnl__all_open_buys,6.0,-71.389244,-0.118982,0.5,smooth_pnl,all selected-wallet open_buy events,False,55,49,0.109091,0.587807,NaN,2.538889,0.207933,0.0
quality_core__score_threshold_0.80,0.0,0.000000,NaN,NaN,quality_core,signal_score >= 0.80,True,1,1,0.000000,NaN,NaN,NaN,NaN,NaN
quality_core__all_open_buys,0.0,0.000000,NaN,NaN,quality_core,all selected-wallet open_buy events,False,3,3,0.000000,NaN,NaN,NaN,NaN,NaN


## Cumulative PnL comparison

In [21]:
from polymarket_analysis.visualization.backtest_plots import plot_strategy_comparison

split_dt = pd.Timestamp(END_DATE_TRAIN) + pd.Timedelta(hours=12)

# calibration bar data for the second panel
# use train_b_scored to build deciles (score_deciles not available in this notebook)
_scored = train_b_scored.copy()
_scored["score_decile"] = pd.qcut(_scored["signal_score"].clip(0, 1), q=10, labels=False, duplicates="drop")
cal_plot = _scored.groupby("score_decile").agg(
    avg_copy_roi_capped=("copy_roi_capped", "mean")
).reset_index()

fig = plot_strategy_comparison(
    strategy_runs,
    strategy_runs_train_ref=strategy_runs_train_ref,
    split_date=split_dt,
    calibration_df=cal_plot,
    title="Wallet signal v2 – strategy comparison",
)
fig.show(renderer="browser")


## Placebo benchmark

In [22]:
rng = np.random.default_rng(42)
eligible_placebo = full_train_metrics[
    (full_train_metrics["open_buy_trades"] >= 20)
    & (full_train_metrics["volume"] >= 500.0)
    & (full_train_metrics.get("distinct_markets", pd.Series(0, index=full_train_metrics.index)) >= 8)
    & (full_train_metrics.get("recent_open_buy_trades", pd.Series(0, index=full_train_metrics.index)) >= 3)
].copy()

SCORE_STRATEGY_NAME = f"quality_core__score_threshold_{SIGNAL_THRESHOLD:.2f}"
test_trades_bt = strategy_runs.get(SCORE_STRATEGY_NAME, {}).get("trades", pd.DataFrame())
test_daily_bt  = strategy_runs.get(SCORE_STRATEGY_NAME, {}).get("daily",  pd.DataFrame())

if len(eligible_placebo) >= len(selected_wallets):
    placebo_wallets = eligible_placebo.sample(n=len(selected_wallets), random_state=42).copy().reset_index(drop=True)
    placebo_wallets["wallet_quality"] = rng.random(len(placebo_wallets))
    placebo_profiles = build_wallet_profiles(
        dataset, placebo_wallets, period="full_train",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
    )
    _, placebo_test_open_buys = build_signal_events(
        dataset, placebo_profiles, period="test",
        end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
        price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
        tx_hash_column=TRIGGER_TX_HASH_COLUMN,
    )
    placebo_scored = apply_signal_score(placebo_test_open_buys, price_table, consensus_table)
    if placebo_scored.empty or "signal_score" not in placebo_scored.columns:
        placebo_compare = pd.DataFrame({"note": ["Placebo signals empty after scoring"]})
    else:
        placebo_trades_bt, placebo_daily_bt, placebo_unfilled_bt, _ = backtest_strategy(
            placebo_scored,
            mask=placebo_scored["signal_score"] >= SIGNAL_THRESHOLD,
            tape_groups=tape_groups,
            strategy_name="placebo_random_wallets",
            trigger_rule=f"placebo signal_score >= {SIGNAL_THRESHOLD:.2f}",
            dynamic_sizing=True,
            cohort_name="placebo",
            **BACKTEST_KWARGS,
        )
        placebo_compare = pd.DataFrame([
            summarize_backtest(test_trades_bt, test_daily_bt).rename(SCORE_STRATEGY_NAME),
            summarize_backtest(placebo_trades_bt, placebo_daily_bt).rename("placebo_random_wallets"),
        ])
        placebo_compare["unfilled_triggers"] = [
            len(trigger_ledgers.get(SCORE_STRATEGY_NAME, pd.DataFrame())),
            len(placebo_unfilled_bt),
        ]
    placebo_compare
else:
    pd.DataFrame({"note": ["Not enough eligible wallets for placebo sample"]})
